해야 할 것들

1) 승객 생존율 예측

2) 첨부 테이터를 이용하여 다양한 머신러닝 모델 적용해보기

3) 첨부 데이터를 다양한 머신러닝 기법에 적용해보기

4) 각 ML 모델의 특성과 사용방법 정리 및 요약

모델 및 라이브러리

1) pdf를 보자.

2) 총 8개의 모델을 비교해야 함

최종 데이터 확인

1) name 열을 5개로 구분하고 숫자형 데이터로 바꾸기 (해결)

2) age 열의 빈 값을 호칭별 중앙값으로 바꾸기 (해결)

3) sex 열에서 male:0, female:1로 바꾸기 (해결)

4) embarked 열 (해결)

5) sibsp, parch 열 (해결)

6) fare 열, 개인별 요금을 나타내는 fare_person 열 추가 (해결)

7) 최종 데이터 확인

1. url에서 파일 다운로드받기

In [1]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report)

In [2]:
# url에서 파일 다운로드

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


2. 데이터 전처리

In [3]:
# 1. 이름을 분류하기
# Mr가 붙으면 0, Miss가 붙으면 1, Mrs가 붙으면 2, Master가 붙으면 3, 아무 표시 없으면 4
df_name = []
for names in df['Name']:
  if 'Mr' in names:
    df_name.append(0)
  elif 'Miss' in names:
    df_name.append(1)
  elif 'Mrs' in names:
    df_name.append(2)
  elif 'Master' in names:
    df_name.append(3)
  else:
    df_name.append(4)

df['Name'] = df_name

In [6]:
# 2. 나이의 결측치를 중앙값으로 채우기
df['Age'] = df['Age'].fillna(df['Age'].median())
print(df['Age'])

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888    28.0
889    26.0
890    32.0
Name: Age, Length: 891, dtype: float64


In [7]:
# 3. 성별 데이터 수치화
df_sex = []
for S in df['Sex']:
    if S == 'male':
        df_sex.append(0)
    else:
        df_sex.append(1)
df['Sex'] = df_sex

In [8]:
# 4. Embarked 열 분류
# S는 0으로, Q는 1로, C는 2로 변환, NULL은 3으로
df_embark = []
for embark in df['Embarked']:
    if embark == 'S':
      df_embark.append(0)
    elif embark == 'Q':
      df_embark.append(1)
    elif embark == 'C':
      df_embark.append(2)
    else:
      df_embark.append(3)
df['Embarked'] = df_embark

In [9]:
# 5. SibSp, Parch -> 이미 int64 형식이라 수정할 필요없음, 결측치도 없음
# 하지만 상식적으로 음수가 나올 수는 없으니 확인차 실행

for A in df['SibSp']:
  if A < 0:
    print(A)
for B in df['Parch']:
  if B < 0:
    print(B)

In [10]:
# 6. fare 열로부터 개인별 요금 fare_person 열 추가
# fare = 탑승 요금
# SibSp = 본인 제외 탑승한 형제자매 및 배우자 수
# Parch = 본인 제외 탑승한 부모 및 자식의 수
# fare_person = 탑승 요금 / (본인 + 형제자매 및 배우자 + 부모 및 자식) = fare / (1 + df['SibSp'] + df['Parch'])

df_fare_person = []
for i in range(len(df)):
  df_fare_person.append(df['Fare'][i] / (df['SibSp'][i] + df['Parch'][i] + 1))
df['Fare_person'] = df_fare_person

In [11]:
#7. Cabin 확인
# 결측치가 더 많아 사용할 수 없음
# 열을 통째로 삭제

df = df.drop('Cabin', axis=1)

In [12]:
#8. Ticket 확인
# 생존과 크게 관련없는 요소
# 열을 통째로 삭제

df = df.drop('Ticket', axis=1)

In [13]:
### 최종 데이터 확인 ###
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    int64  
 4   Sex          891 non-null    int64  
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Fare         891 non-null    float64
 9   Embarked     891 non-null    int64  
 10  Fare_person  891 non-null    float64
dtypes: float64(3), int64(8)
memory usage: 76.7 KB


3. 모델 분석

In [14]:
# 훈련데이터(X_train, y_train)와 테스트데이터(X_test, y_test) 분리
# 테스트데이터 에서는 Survived 열을 일부러 없앤다

X_train, X_test, y_train, y_test = train_test_split(df.drop('Survived', axis=1), df['Survived'], test_size=0.2, random_state=42)

In [15]:
# 실제 정답과 비교해 보는 함수
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

def compare_answer(X_test, y_test, model):
  predictions = model.predict(X_test)

  print("정확도:", accuracy_score(y_test, predictions))   #실제값 y_test와 예측값 predictions 비교
  print("\n혼동행렬", confusion_matrix(y_test, predictions))

In [16]:
#1. 의사결정 나무
# 출력 결과에서 0은 사망, 1은 생존

from sklearn.tree import DecisionTreeClassifier

DT_model = DecisionTreeClassifier(max_depth = 5, random_state = 42)
DT_model.fit(X_train, y_train)
DT_predictions = DT_model.predict(X_test)
print(DT_predictions)


[0 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0
 1 1 0 0 0 0 0 1 0 0 0 0 1 1 1 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 0 1 1 1 0 1
 0 0 1 1 1 0 0 1 1 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 0 0 0 0 1 0 0 1 1 1 0 1 1 0 1 0 1 0 0 0 1 1 1 0 0 0 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 1 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 0 1 0]


In [17]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
DT_accuracy = accuracy_score(y_test, DT_predictions)

compare_answer(X_test, y_test, DT_model)


정확도: 0.7932960893854749

혼동행렬 [[92 13]
 [24 50]]


3-1. Decision Tree 결과

정확도 0.7932960893854749

혼동행렬

[92 13]

[24 50]

In [18]:
# 2. 랜덤 포레스트
from sklearn.ensemble import RandomForestClassifier

RF_model = RandomForestClassifier(n_estimators = 100, max_depth = 10, random_state = 42)
RF_model.fit(X_train, y_train)
RF_predictions = RF_model.predict(X_test)
print(RF_predictions)

[1 0 0 1 0 1 1 0 1 1 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 1 0 0 0
 1 1 0 0 0 0 0 1 0 0 0 0 1 1 1 0 1 0 1 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 1 0 0 0 0 1 0 0 1 1 1 0 0 1 0 1 0 1 0 0 1 1 1 1 0 0 0 0 1 0 0 0 1 0 1
 1 0 0 0 0 1 0 0 1 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 0 0 0 1 1]


In [19]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
RF_accuracy = accuracy_score(y_test, DT_predictions)

compare_answer(X_test, y_test, RF_model)

정확도: 0.8324022346368715

혼동행렬 [[90 15]
 [15 59]]


3-2. Random Forest 결과

정확도 0.8324022346368715

혼동행렬

[90 15]

[15 59]

In [20]:
# 3. 로지스틱 회귀 분석
# max_iter 없이 실행할 경우 lbfgs failed to converge 라고 메시지가 나온다.
# 모델이 학습을 완전히 마치지 못했다는 뜻이다

from sklearn.linear_model import LogisticRegression

LR_model = LogisticRegression(C=1.0, penalty = 'l2', solver = 'lbfgs', random_state = 42)
LR_model.fit(X_train, y_train)
LR_predictions = LR_model.predict(X_test)
print(LR_predictions)

[0 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 1 1 1 0 0 0
 1 1 0 0 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 1 0 0 0 1 0 0 1 1 0 0 0 1 1 1 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 1 1 0 0
 1 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 0 0 1 0 0 1 1 0 1 0 0 0 1 1]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
LR_accuracy = accuracy_score(y_test, LR_predictions)

compare_answer(X_test, y_test, LR_model)

정확도: 0.7821229050279329

혼동행렬 [[86 19]
 [20 54]]


In [22]:
# 모델이 제대로 학습을 끝낸 상태라면?

from sklearn.linear_model import LogisticRegression

LR_FULL_model = LogisticRegression(C=1.0, penalty = 'l2', solver = 'lbfgs', random_state = 42, max_iter = 1000)
LR_FULL_model.fit(X_train, y_train)
LR_FULL_predictions = LR_FULL_model.predict(X_test)
print(LR_FULL_predictions)

[0 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 0 0 1 0 0 1 0 1 1 1 0 0 0
 1 1 0 0 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 0 1 1 1 0 1 1 0 0 1 0 0 0 1 1 1 1 1
 0 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 1 0 1 0 0 0 1 0 0 1 1 1 0 0 1 1 1 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 0 1 1 0 0 1 1 0 1 0 0 0 1 1]


In [23]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
LR_FULL_accuracy = accuracy_score(y_test, LR_FULL_predictions)

compare_answer(X_test, y_test, LR_FULL_model)

정확도: 0.7877094972067039

혼동행렬 [[86 19]
 [19 55]]



3-3. 로지스틱 선형회귀 결과

수렴 안 된 상태/수렴한 상태 비교

정확도: 0.7821229050279329 / 0.7877094972067039

혼동 행렬: [[86 19][20 54]], [[86 19][19 55]]

In [24]:
# 4. 나이브 베이지안

from sklearn.naive_bayes import GaussianNB

NB_model = GaussianNB()
NB_model.fit(X_train, y_train)
NB_predictions = NB_model.predict(X_test)
print(NB_predictions)


[0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 1 0 0 0 0 1 1 1 0 1 0 1 0 1 1 0 0 1 0 0 0 1 1 0 0 1 0 1 0 1
 0 0 0 1 1 0 0 0 1 0 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1
 0 1 0 1 0 1 0 0 0 0 1 1 0 0 0 1 1 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1 1 0 1 0 0 0 1 0]


In [25]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
NB_accuracy = accuracy_score(y_test, NB_predictions)

compare_answer(X_test, y_test, NB_model)

정확도: 0.6871508379888268

혼동행렬 [[88 17]
 [39 35]]


3-4. 나이브 베이지안 결과

정확도 = 0.6871508379888268

혼동행렬 =

[88 17]

[39 35]

In [26]:
# 5. K-NN 모델

from sklearn.neighbors import KNeighborsClassifier

KNN_model = KNeighborsClassifier(n_neighbors=5, metric='minkowski')
KNN_model.fit(X_train, y_train)
KNN_predictions = KNN_model.predict(X_test)

In [27]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
KNN_accuracy = accuracy_score(y_test, KNN_predictions)

compare_answer(X_test, y_test, KNN_model)

정확도: 0.6703910614525139

혼동행렬 [[86 19]
 [40 34]]


3-5. KNN 모델 결과

정확도: 0.6703910614525139

혼동 행렬:

[86 19]

[40 34]

In [28]:
# 6. SVM 모델

from sklearn.svm import LinearSVC

SVM_model = LinearSVC(C = 1.0, random_state = 42)
SVM_model.fit(X_train, y_train)
SVM_predictions = SVM_model.predict(X_test)

In [29]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
SVM_accuracy = accuracy_score(y_test, SVM_predictions)

compare_answer(X_test, y_test, SVM_model)

정확도: 0.7821229050279329

혼동행렬 [[88 17]
 [22 52]]


3-6. SVM 모델 결과

정확도: 0.7821229050279329

혼동 행렬:

[88 17]

[22 52]

In [30]:
# 7. 퍼셉트론 모델

from sklearn.linear_model import Perceptron

PRC_model = Perceptron(max_iter=1000, eta0=1.0, random_state = 42)
PRC_model.fit(X_train, y_train)
PRC_predictions = PRC_model.predict(X_test)



In [31]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
PRC_accuracy = accuracy_score(y_test, PRC_predictions)

compare_answer(X_test, y_test, PRC_model)

정확도: 0.6201117318435754

혼동행렬 [[96  9]
 [59 15]]


3-7. 퍼셉트론 결과

정확도: 0.6201117318435754

혼동행렬:

[96 9]

[59 15]


In [32]:
# 8. 확률적 경사하강법

from sklearn.linear_model import SGDClassifier

SGD_model = SGDClassifier(loss = 'log_loss', penalty = 'l2', random_state=42)
SGD_model.fit(X_train, y_train)
SGD_predictions = SGD_model.predict(X_test)

In [33]:
# 실제 정답과 비교해 보기
# accuracy_score : 테스트 데이터에서 몇프로를 맞췄는지 확인
# confusion matrix : 사망 예측이 맞음 -> 생존을 예측했지만 사망 -> 사망을 예측했지만 생존 -> 생존 예측이 맞음 순으로 출력

# 정확도 비교용
SGD_accuracy = accuracy_score(y_test, SGD_predictions)

compare_answer(X_test, y_test, SGD_model)

정확도: 0.41899441340782123

혼동행렬 [[  2 103]
 [  1  73]]


3-8. 확률적 경사하강법 결과

정확도: 0.41899441340782123

혼동 행렬:

[  2 103]

[  1  73]

4. 모델 비교

In [34]:
accuracy_data = [
    (DT_accuracy, 'Decision Tree'),
    (RF_accuracy, 'Random Forest'),
    (LR_accuracy, 'Logistic Regression (not converged)'),
    (LR_FULL_accuracy, 'Logistic Regression (converged)'),
    (NB_accuracy, 'Naive Bayes'),
    (KNN_accuracy, 'K-NN'),
    (SVM_accuracy, 'SVM'),
    (PRC_accuracy, 'Perceptron'),
    (SGD_accuracy, 'SGD Classifier')
]

# 정확도를 기준으로 정렬합니다.
accuracy_data_sorted = sorted(accuracy_data, key=lambda x: x[0])

# 정렬된 결과를 출력합니다.
for accuracy, model_name in accuracy_data_sorted:
    print(f"{model_name}: {accuracy:.4f}")

print(f"가장 정확도가 높은 모델은 {accuracy_data_sorted[8]}입니다.")

SGD Classifier: 0.4190
Perceptron: 0.6201
K-NN: 0.6704
Naive Bayes: 0.6872
Logistic Regression (not converged): 0.7821
SVM: 0.7821
Logistic Regression (converged): 0.7877
Decision Tree: 0.7933
Random Forest: 0.7933
가장 정확도가 높은 모델은 (0.7932960893854749, 'Random Forest')입니다.


5. 반복학습을 한다면?

In [35]:
import time
from tqdm import tqdm

#반복학습 후 모델을 반환하는 함수
def repeat_learn(model):
  repeat_model = []
  for epoch in tqdm(range(10), desc = "[학습 중]"):
    model.fit(X_train, y_train)
    time.sleep(0.5)
  print("학습이 완료되었습니다.")
  repeat_model = model
  return repeat_model


In [36]:
DT_repeat = repeat_learn(DT_model)
compare_answer(X_test, y_test, DT_repeat)

[학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.96it/s]

학습이 완료되었습니다.
정확도: 0.7932960893854749

혼동행렬 [[92 13]
 [24 50]]


In [37]:
RF_repeat = repeat_learn(RF_model)
compare_answer(X_test, y_test, RF_repeat)

[학습 중]: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s]

학습이 완료되었습니다.
정확도: 0.8324022346368715

혼동행렬 [[90 15]
 [15 59]]


In [38]:
LR_repeat = repeat_learn(LR_model)
compare_answer(X_test, y_test, LR_repeat)

[학습 중]:   0%|          | 0/10 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[학습 중]:  10%|█         | 1/10 [00:00<00:04,  1.90it/s]/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver option

학습이 완료되었습니다.
정확도: 0.7821229050279329

혼동행렬 [[86 19]
 [20 54]]


In [39]:
LR_FULL_repeat = repeat_learn(LR_FULL_model)
compare_answer(X_test, y_test, LR_FULL_repeat)

[학습 중]: 100%|██████████| 10/10 [00:07<00:00,  1.39it/s]

학습이 완료되었습니다.
정확도: 0.7877094972067039

혼동행렬 [[86 19]
 [19 55]]


In [40]:
NB_repeat = repeat_learn(NB_model)
compare_answer(X_test, y_test, NB_repeat)

[학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.97it/s]

학습이 완료되었습니다.
정확도: 0.6871508379888268

혼동행렬 [[88 17]
 [39 35]]


In [41]:
KNN_repeat = repeat_learn(KNN_model)
compare_answer(X_test, y_test, KNN_repeat)

[학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.89it/s]

학습이 완료되었습니다.
정확도: 0.6703910614525139

혼동행렬 [[86 19]
 [40 34]]


In [42]:
SVM_repeat = repeat_learn(SVM_model)
compare_answer(X_test, y_test, SVM_repeat)

[학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.93it/s]

학습이 완료되었습니다.
정확도: 0.7821229050279329

혼동행렬 [[88 17]
 [22 52]]


In [43]:
PRC_repeat = repeat_learn(PRC_model)
compare_answer(X_test, y_test, PRC_repeat)

[학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.97it/s]

학습이 완료되었습니다.
정확도: 0.6201117318435754

혼동행렬 [[96  9]
 [59 15]]


In [44]:
SGD_repeat = repeat_learn(SGD_model)
compare_answer(X_test, y_test, SGD_repeat)

[학습 중]: 100%|██████████| 10/10 [00:05<00:00,  1.95it/s]

학습이 완료되었습니다.
정확도: 0.41899441340782123

혼동행렬 [[  2 103]
 [  1  73]]


In [50]:
# 반복학습 모델 accuracy 출력모델
def accuracy_return(model, X_test, y_test):
  accuracy = accuracy_score(y_test, model.predict(X_test))
  return accuracy

# 반복 학습된 모델들과 그에 해당하는 이름 리스트
repeat_model_data = [
    (accuracy_return(DT_repeat, X_test, y_test), 'Decision Tree (Repeat)'),
    (accuracy_return(RF_repeat, X_test, y_test), 'Random Forest (Repeat)'),
    (accuracy_return(LR_repeat, X_test, y_test), 'Logistic Regression (not converged, Repeat)'),
    (accuracy_return(LR_FULL_repeat, X_test, y_test), 'Logistic Regression (converged, Repeat)'),
    (accuracy_return(NB_repeat, X_test, y_test), 'Naive Bayes (Repeat)'),
    (accuracy_return(KNN_repeat, X_test, y_test), 'K-NN (Repeat)'),
    (accuracy_return(SVM_repeat, X_test, y_test), 'SVM (Repeat)'),
    (accuracy_return(PRC_repeat, X_test, y_test), 'Perceptron (Repeat)'),
    (accuracy_return(SGD_repeat, X_test, y_test), 'SGD Classifier (Repeat)')
]

# 정확도를 기준으로 정렬합니다.
accuracy_data_repeat_sorted = sorted(repeat_model_data, key=lambda x: x[0])

# 정렬된 결과를 출력합니다.
for accuracy, model_name in accuracy_data_repeat_sorted:
    print(f"{model_name}: {accuracy:.4f}")

# 가장 정확도가 높은 모델 출력
highest_accuracy_model = accuracy_data_repeat_sorted[-1]
print(f"가장 정확도가 높은 모델은 {highest_accuracy_model[1]}이며, 정확도는 {highest_accuracy_model[0]:.4f}입니다.")


SGD Classifier (Repeat): 0.4190
Perceptron (Repeat): 0.6201
K-NN (Repeat): 0.6704
Naive Bayes (Repeat): 0.6872
Logistic Regression (not converged, Repeat): 0.7821
SVM (Repeat): 0.7821
Logistic Regression (converged, Repeat): 0.7877
Decision Tree (Repeat): 0.7933
Random Forest (Repeat): 0.8324
가장 정확도가 높은 모델은 Random Forest (Repeat)이며, 정확도는 0.8324입니다.
